# 01 - Setup + Extract (frozen backbone -> logits + penultimate embeddings)

Runner only; logic in `pcc/extract`, `pcc/data`. **Blocked by the Phase-0 gate:** this
notebook refuses to run unless `00_verify_checkpoint` wrote a GATE_PASSED marker for this
(dataset, split) on Drive (AGENTS.md; release_audit.md). Extracting from an unverified
checkpoint produces silently-meaningless embeddings.

**Pass criteria (pre-registered):** `(logits, embeddings)` shards + a checksummed manifest
written to Drive; resumable across sessions (Sec 3.1); embeddings extracted only for the
gated backbone.


## 1. GPU check


In [ ]:
import subprocess
o = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(o.stdout if o.returncode==0 else 'WARNING: no GPU - autobatch falls back to CPU (slow).')


## 2. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'plantnet'          # must match a passed gate
BACKBONE   = 'resnet50_ltc'
SPLIT      = 'val'               # val (full) | train_subset (per-class quota)
PER_CLASS_QUOTA = None           # set for train_subset; fixed by 01_descriptor_stability
SEED, ALPHA = 42, 0.1
CHECKPOINT_EVERY = 50            # batches between Drive checkpoints (Sec 3.1)
OUT_DIR    = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}/{SPLIT}'
# =======================================================================
print('OUT_DIR =', OUT_DIR)


## 3. Mount Drive + repo + pinned env + seed + versions


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
os.environ['PYTHONPATH'] = REPO_ROOT + os.pathsep + os.environ.get('PYTHONPATH','')
os.environ['PYTHONUTF8'] = '1'
from pcc.utils.seed import set_seed
from pcc.utils.device import gpu_name
from pcc.utils.io import environment_stamp
set_seed(SEED)
print('GPU:', gpu_name()); print('env:', environment_stamp()['packages'])


## 4. GATE CHECK - refuse to run unless the Phase-0 gate passed


In [ ]:
import os, json
GATE_MARKER = f'{DRIVE_ROOT}/gates/GATE_PASSED_{DATASET}_{SPLIT}.json'
if not os.path.exists(GATE_MARKER):
    raise SystemExit(f'BLOCKED: no gate marker at {GATE_MARKER}. Run 00_verify_checkpoint '
                     f'for ({DATASET},{SPLIT}) first and get a PASS. (release_audit.md)')
gate = json.load(open(GATE_MARKER))
print('gate OK:', gate['results']['verdict'], '| ckpt sha256:',
      gate['checksums']['checkpoint'][:16], '...')


## 5. Resume check - read manifest before computing anything (Sec 3.1)


In [ ]:
# TODO(extract): from pcc.extract.forward import resume_point
# start_batch = resume_point(OUT_DIR)   # 0 if fresh; else continue from Drive
print('resume point: TODO once pcc/extract/forward.py (sharded writer/manifest) exists')


## 6. Extract logits + penultimate embeddings (single pass, autobatch, checkpointed)


In [ ]:
# Uses the SAME gated checkpoint. Reuse pcc.extract.backbones for the forward pass;
# add a sharded, resumable writer (pcc/extract/forward.py) that checkpoints every
# CHECKPOINT_EVERY batches to OUT_DIR and writes a checksummed manifest (pcc/data/manifest.py).
# from pcc.extract.backbones import load_ltc_resnet50, forward_logits_and_embeddings
# from pcc.data.ltc_datasets import test_transform, plantnet_val / INaturalist2018Val
print('extraction: implement pcc/extract/forward.py (sharded+resumable), then wire here')


## 7. Write report


In [ ]:
import time
from pcc.utils.io import write_report
# write_report('pcc/reports', f'01_extract_{DATASET}_{SPLIT}',
#   hypothesis='embeddings+logits extracted from gated checkpoint, manifest checksummed',
#   pass_criteria='all shards written; manifest verifies; resumable; gated backbone only',
#   config=dict(dataset=DATASET, backbone=BACKBONE, split=SPLIT, quota=PER_CLASS_QUOTA),
#   seed=SEED, results={}, conclusion='TODO', started_at=time.time())
print('report: wire once extraction returns results')
